In [7]:
import os, random, time, json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from imblearn.metrics import specificity_score
from mambapy.vim import VMamba, MambaConfig
from thop import profile

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [8]:
AUG_ROOT = "D:/mamba_model/aug_clean_tio"       
TAG      = "tio_tf"                                  
# ───────────────────────────────────────────────────────────────

COHORT_CSV = "D:/mamba_model/thesis_cohort_clean.csv"
MRI_CACHE  = f"{AUG_ROOT}/roi_mri"
PET_CACHE  = f"{AUG_ROOT}/roi_pet"
CKPT_DIR   = f"D:/mamba_model/checkpoints_v7_roi_{TAG}"
RESULTS    = f"D:/mamba_model/v7_roi_{TAG}_results.json"
os.makedirs(CKPT_DIR, exist_ok=True)

SPLIT_SEED  = 42
AUG_SEEDS   = [1, 101, 42]
BATCH_SIZE  = 4
NUM_WORKERS = 0          

print(f"aug:  {AUG_ROOT}")
print(f"ckpt: {CKPT_DIR}")
for p in (MRI_CACHE, PET_CACHE):
    n = len(os.listdir(p)) if os.path.isdir(p) else 0
    print(f"  {os.path.basename(p)}: {n} files{'  *** MISSING ***' if n == 0 else ''}")

aug:  D:/mamba_model/aug_clean_tio
ckpt: D:/mamba_model/checkpoints_v7_roi_tio_tf
  roi_mri: 560 files
  roi_pet: 560 files


In [9]:
class TransformerEncoderBlock(nn.Module):
    """Replaces VimEncoder. Same d_model and n_layers, so the comparison
    against Mamba is at matched depth and width

    Note: attention is O(n^2) against Mamba's linear scan."""
    def __init__(self, d_model=32, n_layers=2, n_heads=4, dropout=0.1):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, tokens):
        return self.final_norm(self.encoder(tokens))

In [10]:
class ROIPatchEmbed3D(nn.Module):
    """6 ROIs -> non-overlapping 8^3 patches -> one token each. 3072 tokens."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32):
        super().__init__()
        self.n_rois, self.patch_size = n_rois, patch_size
        self.grid_size = roi_size // patch_size
        self.patches_per_roi = self.grid_size ** 3
        self.d_model = d_model
        self.patch_conv = nn.Conv3d(1, d_model, kernel_size=patch_size, stride=patch_size)
        self.roi_embed    = nn.Embedding(n_rois, d_model)
        self.depth_embed  = nn.Embedding(self.grid_size, d_model)
        self.height_embed = nn.Embedding(self.grid_size, d_model)
        self.width_embed  = nn.Embedding(self.grid_size, d_model)
        with torch.no_grad():
            for e in [self.roi_embed, self.depth_embed, self.height_embed, self.width_embed]:
                e.weight.mul_(0.02)
        d, h, w = torch.meshgrid(torch.arange(self.grid_size), torch.arange(self.grid_size),
                                 torch.arange(self.grid_size), indexing="ij")
        self.register_buffer("coordinates", torch.stack([d, h, w], -1).reshape(-1, 3),
                             persistent=False)

    def forward(self, rois):
        B, n = rois.shape[:2]
        x = rois.reshape(B * n, 1, *rois.shape[-3:])
        tokens = self.patch_conv(x).flatten(2).transpose(1, 2)
        tokens = tokens.reshape(B, n, self.patches_per_roi, self.d_model)
        c = self.coordinates
        spatial = (self.depth_embed(c[:, 0]) + self.height_embed(c[:, 1])
                   + self.width_embed(c[:, 2]))
        tokens = tokens + spatial[None, None] + self.roi_embed.weight[None, :, None, :]
        occ = F.max_pool3d((x.abs() > 1e-6).float(),
                           kernel_size=self.patch_size, stride=self.patch_size)
        valid = occ.flatten(1).bool().reshape(B, n, self.patches_per_roi)
        tokens = tokens.reshape(B, -1, self.d_model)
        valid = valid.reshape(B, -1)
        return tokens * valid.unsqueeze(-1).to(tokens.dtype), valid


class TransformerEncoderBlock(nn.Module):
    """Drop-in for VimEncoder. Same d_model and n_layers so the comparison
    is at matched depth and width."""
    def __init__(self, d_model=32, n_layers=2, n_heads=4, dropout=0.1):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.final_norm = nn.LayerNorm(d_model)
    def forward(self, tokens):
        return self.final_norm(self.encoder(tokens))


class TransformerBranch(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32,
                 n_layers=2, d_state=16):
        super().__init__()
        self.n_rois = n_rois
        self.patch_embed = ROIPatchEmbed3D(n_rois, roi_size, patch_size, d_model)
        self.encoder = TransformerEncoderBlock(d_model, n_layers)
    def forward(self, rois):
        tokens, valid = self.patch_embed(rois)
        tokens = self.encoder(tokens)
        w = valid.unsqueeze(-1).to(tokens.dtype)
        return (tokens * w).sum(1) / w.sum(1).clamp_min(1.0)


class TransformerModel(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32,
                 n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.branch = TransformerBranch(n_rois, roi_size, patch_size, d_model, n_layers)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, n_classes)
    def forward(self, rois):
        return self.classifier(self.dropout(self.branch(rois)))


class MultimodalTransformerModel(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32,
                 n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.mri_branch = TransformerBranch(n_rois, roi_size, patch_size, d_model, n_layers)
        self.pet_branch = TransformerBranch(n_rois, roi_size, patch_size, d_model, n_layers)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model * 2, n_classes)
    def forward(self, mri, pet):
        f = torch.cat([self.mri_branch(mri), self.pet_branch(pet)], dim=1)
        return self.classifier(self.dropout(f))


_m = TransformerModel()
print(f"params {sum(p.numel() for p in _m.parameters()):,} "
      f"(Mamba equivalent was 44,962)")
del _m

params 42,914 (Mamba equivalent was 44,962)


C:\Users\sammy\miniconda3\envs\mamba_thesis\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [15]:
df = pd.read_csv(COHORT_CSV)
sessions, labels = df["mri_session"].values, df["outcome_label"].values

X_tv, X_test, y_tv, y_test = train_test_split(
    sessions, labels, test_size=0.2, random_state=SPLIT_SEED, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=SPLIT_SEED, stratify=y_tv)

session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))
print(f"train {len(X_train)} | val {len(X_val)} | test {len(X_test)} "
      f"| test pos {int(y_test.sum())}")


# ── in-memory cache: first epoch reads from disk, the rest from RAM 
# ~3.4 GB per modality (560 files x 6 MB)
_CACHE = {}

def load_cached(path):
    a = _CACHE.get(path)
    if a is None:
        a = np.load(path).astype(np.float32)
        _CACHE[path] = a
    return a
# NOTE: torch.from_numpy shares memory with the cached array.


class ROIDataset(Dataset):
    """Single modality. Training set includes 3 augmented copies per subject."""
    def __init__(self, sessions, labels, cache_dir, is_mri=True, is_train=False):
        self.samples, self.cache_dir = [], cache_dir
        for ses, lab in zip(sessions, labels):
            key = ses if is_mri else session_to_subject[ses]
            self.samples.append((key, lab, "orig"))
            if is_train:
                for s in AUG_SEEDS:
                    self.samples.append((key, lab, f"aug{s}"))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        key, lab, ver = self.samples[i]
        a = load_cached(f"{self.cache_dir}/{key}_{ver}.npy")
        return torch.from_numpy(a).unsqueeze(1), torch.tensor(lab, dtype=torch.long), key


class MultimodalROIDataset(Dataset):
    """Pairs MRI and PET for the same subject and the same augmentation seed."""
    def __init__(self, sessions, labels, mri_dir, pet_dir, is_train=False):
        self.samples, self.mri_dir, self.pet_dir = [], mri_dir, pet_dir
        for ses, lab in zip(sessions, labels):
            sid = session_to_subject[ses]
            self.samples.append((ses, sid, lab, "orig"))
            if is_train:
                for s in AUG_SEEDS:
                    self.samples.append((ses, sid, lab, f"aug{s}"))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        mk, pk, lab, ver = self.samples[i]
        m = load_cached(f"{self.mri_dir}/{mk}_{ver}.npy")
        p = load_cached(f"{self.pet_dir}/{pk}_{ver}.npy")
        return (torch.from_numpy(m).unsqueeze(1), torch.from_numpy(p).unsqueeze(1),
                torch.tensor(lab, dtype=torch.long), mk)


def dl(ds, shuffle=False):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      persistent_workers=(NUM_WORKERS > 0))

mri_loaders = (dl(ROIDataset(X_train, y_train, MRI_CACHE, True, True), True),
               dl(ROIDataset(X_val,   y_val,   MRI_CACHE, True, False)),
               dl(ROIDataset(X_test,  y_test,  MRI_CACHE, True, False)))

pet_loaders = (dl(ROIDataset(X_train, y_train, PET_CACHE, False, True), True),
               dl(ROIDataset(X_val,   y_val,   PET_CACHE, False, False)),
               dl(ROIDataset(X_test,  y_test,  PET_CACHE, False, False)))

mm_loaders  = (dl(MultimodalROIDataset(X_train, y_train, MRI_CACHE, PET_CACHE, True), True),
               dl(MultimodalROIDataset(X_val,   y_val,   MRI_CACHE, PET_CACHE, False)),
               dl(MultimodalROIDataset(X_test,  y_test,  MRI_CACHE, PET_CACHE, False)))

print(f"train samples (4x augmented): {len(mri_loaders[0].dataset)}")

# first pass fills the cache from disk
t0 = time.time()
for i, _ in enumerate(mri_loaders[0]):
    if i >= 20: break
cold = time.time() - t0

t0 = time.time()
for i, _ in enumerate(mri_loaders[0]):
    if i >= 20: break
warm = time.time() - t0

print(f"20 batches: cold {cold:.1f}s -> warm {warm:.1f}s")
print(f"cached arrays: {len(_CACHE)}  (~{sum(a.nbytes for a in _CACHE.values())/1e9:.1f} GB)")

train 120 | val 40 | test 40 | test pos 20
train samples (4x augmented): 480
20 batches: cold 4.1s -> warm 3.5s
cached arrays: 156  (~1.0 GB)


In [16]:
def train_epoch(model, loader, opt, crit, mm):
    model.train(); tot = 0
    for batch in loader:
        opt.zero_grad()
        if mm:
            a, b, lb, _ = batch; out = model(a.to(device), b.to(device))
        else:
            a, lb, _ = batch;    out = model(a.to(device))
        loss = crit(out, lb.to(device)); loss.backward(); opt.step(); tot += loss.item()
    return tot / len(loader)

def evaluate(model, loader, crit, mm):
    model.eval(); tot, P, L = 0, [], []
    with torch.no_grad():
        for batch in loader:
            if mm:
                a, b, lb, _ = batch; out = model(a.to(device), b.to(device))
            else:
                a, lb, _ = batch;    out = model(a.to(device))
            tot += crit(out, lb.to(device)).item()
            P.extend(out.argmax(1).cpu().numpy()); L.extend(lb.numpy())
    return (tot/len(loader), np.mean(np.array(P) == np.array(L)),
            recall_score(L, P, zero_division=0), specificity_score(L, P))

def measure_inference(model, loader, mm, n=20):
    model.eval(); ts = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n: break
            if mm:
                a, b = batch[0].to(device), batch[1].to(device); bs = a.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time(); _ = model(a, b)
            else:
                a = batch[0].to(device); bs = a.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time(); _ = model(a)
            if device.type == 'cuda': torch.cuda.synchronize()
            ts.append((time.time() - t0) / bs)
    return np.mean(ts), np.std(ts)

def compute_flops(model, loader, mm):
    try:
        model.eval(); b = next(iter(loader))
        with torch.no_grad():
            inp = (b[0][:1].to(device), b[1][:1].to(device)) if mm else (b[0][:1].to(device),)
            macs, _ = profile(model, inputs=inp, verbose=False)
        return macs * 2
    except Exception as e:
        print(f"  (FLOPs failed: {e})"); return None


def run_seed(seed, model_cls, loaders, mm, prefix,
             max_epochs=101, patience=15, min_epochs=25, lr=1e-4, log_every=5):
    torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    np.random.seed(seed); random.seed(seed)
    tr, va, te = loaders

    model = model_cls(d_model=32, n_layers=2, n_classes=2, dropout=0.4).to(device)
    crit  = nn.CrossEntropyLoss(label_smoothing=0.05)
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    sch   = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=10)

    best, no_imp, best_ep, total = float('inf'), 0, 0, 0
    path = f"{CKPT_DIR}/{prefix}_seed{seed}.pt"
    print(f"\n--- {prefix} seed {seed} ---")

    for ep in range(1, max_epochs):
        t0 = time.time()
        trl = train_epoch(model, tr, opt, crit, mm)
        vl, vacc, vtpr, vtnr = evaluate(model, va, crit, mm)
        sch.step(vl); dt = time.time() - t0; total += dt
        if ep % log_every == 0 or ep == 1:
            print(f"  ep {ep:>3} | train {trl:.4f} | val {vl:.4f} | "
                  f"acc {vacc:.3f} tpr {vtpr:.3f} tnr {vtnr:.3f} | {dt:.0f}s")
        if vl < best:
            best, best_ep, no_imp = vl, ep, 0
            torch.save(model.state_dict(), path)
        else:
            no_imp += 1
            if ep >= min_epochs and no_imp >= patience:
                print(f"  early stop {ep}, best {best_ep}"); break

    if best_ep < 5:
        print(f"  WARNING: best epoch {best_ep} -- may not have trained")

    model.load_state_dict(torch.load(path, weights_only=True))
    _, acc, tpr, tnr = evaluate(model, te, crit, mm)
    npar = sum(p.numel() for p in model.parameters() if p.requires_grad)
    inf_m, inf_s = measure_inference(model, te, mm)
    fl = compute_flops(model, te, mm)

    print(f"  >>> TEST Acc={acc*100:.1f}% TPR={tpr*100:.1f}% TNR={tnr*100:.1f}% | "
          f"params={npar:,} train={total/60:.1f}min inf={inf_m*1000:.2f}ms "
          f"{f'{fl/1e9:.2f}GFLOPs' if fl else ''} best_ep={best_ep}")

    return {"seed": seed, "acc": acc, "tpr": tpr, "tnr": tnr, "best_epoch": best_ep,
            "train_time_sec": total, "n_params": npar,
            "inf_time_ms": inf_m*1000, "flops": fl}

results = {"mri": [], "pet": [], "mm": []}

In [7]:
results["mri"].append(run_seed(1, TransformerModel, mri_loaders, False, "v7_roi_tf_mri"))

C:\Users\sammy\miniconda3\envs\mamba_thesis\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(



--- v7_roi_tf_mri seed 1 ---
  ep   1 | train 0.7273 | val 0.7053 | acc 0.500 tpr 1.000 tnr 0.000 | 27s
  ep   5 | train 0.6973 | val 0.6870 | acc 0.500 tpr 0.900 tnr 0.100 | 8s
  ep  10 | train 0.6964 | val 0.6869 | acc 0.500 tpr 0.000 tnr 1.000 | 8s
  ep  15 | train 0.6922 | val 0.6842 | acc 0.525 tpr 0.050 tnr 1.000 | 7s
  ep  20 | train 0.6826 | val 0.6852 | acc 0.500 tpr 1.000 tnr 0.000 | 7s
  ep  25 | train 0.6406 | val 0.6505 | acc 0.575 tpr 0.200 tnr 0.950 | 7s
  ep  30 | train 0.6121 | val 0.6124 | acc 0.675 tpr 0.600 tnr 0.750 | 7s
  ep  35 | train 0.5699 | val 0.6196 | acc 0.675 tpr 0.650 tnr 0.700 | 7s
  ep  40 | train 0.5169 | val 0.7280 | acc 0.600 tpr 0.450 tnr 0.750 | 7s
  ep  45 | train 0.4450 | val 0.8497 | acc 0.675 tpr 0.450 tnr 0.900 | 7s
  early stop 45, best 30
  >>> TEST Acc=55.0% TPR=50.0% TNR=60.0% | params=42,914 train=5.9min inf=6.55ms 0.21GFLOPs best_ep=30


In [8]:
results["mri"].append(run_seed(7, TransformerModel, mri_loaders, False, "v7_roi_tf_mri"))


--- v7_roi_tf_mri seed 7 ---


C:\Users\sammy\miniconda3\envs\mamba_thesis\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


  ep   1 | train 0.7132 | val 0.6935 | acc 0.475 tpr 0.950 tnr 0.000 | 7s
  ep   5 | train 0.6998 | val 0.6919 | acc 0.600 tpr 0.200 tnr 1.000 | 7s
  ep  10 | train 0.6961 | val 0.7064 | acc 0.500 tpr 1.000 tnr 0.000 | 7s
  ep  15 | train 0.6893 | val 0.6887 | acc 0.575 tpr 0.450 tnr 0.700 | 7s
  ep  20 | train 0.6939 | val 0.6873 | acc 0.525 tpr 1.000 tnr 0.050 | 7s
  ep  25 | train 0.6747 | val 0.6854 | acc 0.500 tpr 1.000 tnr 0.000 | 7s
  ep  30 | train 0.6288 | val 0.6205 | acc 0.675 tpr 0.650 tnr 0.700 | 7s
  ep  35 | train 0.5669 | val 0.6781 | acc 0.650 tpr 0.700 tnr 0.600 | 7s
  ep  40 | train 0.5108 | val 0.6880 | acc 0.625 tpr 0.650 tnr 0.600 | 7s
  ep  45 | train 0.4099 | val 0.8230 | acc 0.650 tpr 0.750 tnr 0.550 | 7s
  early stop 45, best 30
  >>> TEST Acc=55.0% TPR=40.0% TNR=70.0% | params=42,914 train=5.5min inf=3.23ms 0.21GFLOPs best_ep=30


In [17]:
results["mri"].append(run_seed(123, TransformerModel, mri_loaders, False, "v7_roi_tf_mri"))

C:\Users\sammy\miniconda3\envs\mamba_thesis\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(



--- v7_roi_tf_mri seed 123 ---
  ep   1 | train 0.6992 | val 0.6985 | acc 0.500 tpr 1.000 tnr 0.000 | 25s
  ep   5 | train 0.6910 | val 0.6942 | acc 0.525 tpr 0.900 tnr 0.150 | 7s
  ep  10 | train 0.6875 | val 0.6928 | acc 0.500 tpr 1.000 tnr 0.000 | 8s
  ep  15 | train 0.6579 | val 0.6543 | acc 0.600 tpr 0.600 tnr 0.600 | 8s
  ep  20 | train 0.6067 | val 0.6829 | acc 0.600 tpr 0.350 tnr 0.850 | 8s
  ep  25 | train 0.5343 | val 0.7097 | acc 0.625 tpr 0.800 tnr 0.450 | 7s
  ep  30 | train 0.4513 | val 0.8231 | acc 0.625 tpr 0.600 tnr 0.650 | 7s
  early stop 33, best 18
  >>> TEST Acc=47.5% TPR=45.0% TNR=50.0% | params=42,914 train=4.4min inf=5.95ms 0.21GFLOPs best_ep=18


In [18]:
results["pet"].append(run_seed(1, TransformerModel, pet_loaders, False, "v7_roi_tf_pet"))

C:\Users\sammy\miniconda3\envs\mamba_thesis\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(



--- v7_roi_tf_pet seed 1 ---
  ep   1 | train 0.7226 | val 0.6850 | acc 0.500 tpr 1.000 tnr 0.000 | 44s
  ep   5 | train 0.6514 | val 0.6401 | acc 0.675 tpr 0.400 tnr 0.950 | 7s
  ep  10 | train 0.6352 | val 0.6500 | acc 0.600 tpr 0.250 tnr 0.950 | 7s
  ep  15 | train 0.6302 | val 0.6371 | acc 0.625 tpr 0.300 tnr 0.950 | 7s
  ep  20 | train 0.6160 | val 0.6458 | acc 0.600 tpr 0.950 tnr 0.250 | 7s
  ep  25 | train 0.5394 | val 0.6273 | acc 0.675 tpr 0.650 tnr 0.700 | 7s
  ep  30 | train 0.5357 | val 0.6032 | acc 0.700 tpr 0.700 tnr 0.700 | 7s
  early stop 34, best 19
  >>> TEST Acc=72.5% TPR=65.0% TNR=80.0% | params=42,914 train=4.8min inf=7.05ms 0.21GFLOPs best_ep=19


In [19]:
results["pet"].append(run_seed(7, TransformerModel, pet_loaders, False, "v7_roi_tf_pet"))


--- v7_roi_tf_pet seed 7 ---


C:\Users\sammy\miniconda3\envs\mamba_thesis\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


  ep   1 | train 0.7026 | val 0.6748 | acc 0.675 tpr 0.500 tnr 0.850 | 7s
  ep   5 | train 0.6562 | val 0.6297 | acc 0.675 tpr 0.400 tnr 0.950 | 7s
  ep  10 | train 0.6271 | val 0.6428 | acc 0.600 tpr 1.000 tnr 0.200 | 7s
  ep  15 | train 0.6023 | val 0.6053 | acc 0.675 tpr 0.450 tnr 0.900 | 7s
  ep  20 | train 0.5901 | val 0.6636 | acc 0.600 tpr 1.000 tnr 0.200 | 7s
  ep  25 | train 0.5883 | val 0.6397 | acc 0.625 tpr 0.850 tnr 0.400 | 7s
  ep  30 | train 0.4953 | val 0.6767 | acc 0.625 tpr 0.800 tnr 0.450 | 7s
  early stop 32, best 17
  >>> TEST Acc=70.0% TPR=65.0% TNR=75.0% | params=42,914 train=3.9min inf=3.27ms 0.21GFLOPs best_ep=17


In [20]:
results["pet"].append(run_seed(123, TransformerModel, pet_loaders, False, "v7_roi_tf_pet"))


--- v7_roi_tf_pet seed 123 ---


C:\Users\sammy\miniconda3\envs\mamba_thesis\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


  ep   1 | train 0.6970 | val 0.6861 | acc 0.500 tpr 1.000 tnr 0.000 | 7s
  ep   5 | train 0.6646 | val 0.6355 | acc 0.700 tpr 0.450 tnr 0.950 | 7s
  ep  10 | train 0.6170 | val 0.6098 | acc 0.650 tpr 0.800 tnr 0.500 | 7s
  ep  15 | train 0.5558 | val 0.6158 | acc 0.700 tpr 0.450 tnr 0.950 | 7s
  ep  20 | train 0.4870 | val 0.6021 | acc 0.725 tpr 0.650 tnr 0.800 | 7s
  ep  25 | train 0.3403 | val 0.7946 | acc 0.650 tpr 0.700 tnr 0.600 | 7s
  ep  30 | train 0.1996 | val 0.9697 | acc 0.600 tpr 0.550 tnr 0.650 | 7s
  early stop 32, best 17
  >>> TEST Acc=60.0% TPR=50.0% TNR=70.0% | params=42,914 train=3.9min inf=3.19ms 0.21GFLOPs best_ep=17


In [21]:
results["mm"].append(run_seed(1, MultimodalTransformerModel, mm_loaders, True, "v7_roi_mm"))


--- v7_roi_mm seed 1 ---


C:\Users\sammy\miniconda3\envs\mamba_thesis\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


  ep   1 | train 0.7376 | val 0.6853 | acc 0.500 tpr 0.000 tnr 1.000 | 15s
  ep   5 | train 0.6495 | val 0.6631 | acc 0.575 tpr 0.200 tnr 0.950 | 15s
  ep  10 | train 0.6181 | val 0.6471 | acc 0.625 tpr 0.300 tnr 0.950 | 15s
  ep  15 | train 0.6031 | val 0.6155 | acc 0.700 tpr 0.600 tnr 0.800 | 15s
  ep  20 | train 0.5450 | val 0.6822 | acc 0.625 tpr 0.300 tnr 0.950 | 15s
  ep  25 | train 0.5285 | val 0.5648 | acc 0.700 tpr 0.700 tnr 0.700 | 15s
  ep  30 | train 0.4561 | val 0.6327 | acc 0.700 tpr 0.700 tnr 0.700 | 15s
  ep  35 | train 0.3069 | val 0.6927 | acc 0.675 tpr 0.650 tnr 0.700 | 15s
  ep  40 | train 0.1985 | val 0.8487 | acc 0.700 tpr 0.700 tnr 0.700 | 15s
  early stop 40, best 25
  >>> TEST Acc=62.5% TPR=55.0% TNR=70.0% | params=85,826 train=9.8min inf=6.38ms 0.41GFLOPs best_ep=25


In [22]:
results["mm"].append(run_seed(7, MultimodalTransformerModel, mm_loaders, True, "v7_roi_tf_mm"))


--- v7_roi_tf_mm seed 7 ---


C:\Users\sammy\miniconda3\envs\mamba_thesis\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


  ep   1 | train 0.7206 | val 0.6730 | acc 0.675 tpr 0.450 tnr 0.900 | 15s
  ep   5 | train 0.6591 | val 0.6452 | acc 0.700 tpr 0.450 tnr 0.950 | 15s
  ep  10 | train 0.6348 | val 0.6530 | acc 0.650 tpr 0.350 tnr 0.950 | 15s
  ep  15 | train 0.6161 | val 0.6390 | acc 0.675 tpr 0.400 tnr 0.950 | 15s
  ep  20 | train 0.5915 | val 0.6709 | acc 0.525 tpr 0.950 tnr 0.100 | 16s
  ep  25 | train 0.5064 | val 0.6143 | acc 0.725 tpr 0.650 tnr 0.800 | 15s
  ep  30 | train 0.4764 | val 0.5896 | acc 0.725 tpr 0.700 tnr 0.750 | 15s
  ep  35 | train 0.3259 | val 0.7205 | acc 0.725 tpr 0.600 tnr 0.850 | 15s
  early stop 38, best 23
  >>> TEST Acc=52.5% TPR=60.0% TNR=45.0% | params=85,826 train=9.6min inf=6.45ms 0.41GFLOPs best_ep=23


In [23]:
results["mm"].append(run_seed(123, MultimodalTransformerModel, mm_loaders, True, "v7_roi_tf_mm"))


--- v7_roi_tf_mm seed 123 ---


C:\Users\sammy\miniconda3\envs\mamba_thesis\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


  ep   1 | train 0.7125 | val 0.6666 | acc 0.500 tpr 1.000 tnr 0.000 | 15s
  ep   5 | train 0.6696 | val 0.6276 | acc 0.700 tpr 0.450 tnr 0.950 | 15s
  ep  10 | train 0.6251 | val 0.6358 | acc 0.750 tpr 0.650 tnr 0.850 | 19s
  ep  15 | train 0.6049 | val 0.6258 | acc 0.675 tpr 0.400 tnr 0.950 | 15s
  ep  20 | train 0.5475 | val 0.5925 | acc 0.725 tpr 0.600 tnr 0.850 | 17s
  ep  25 | train 0.5095 | val 0.5883 | acc 0.775 tpr 0.700 tnr 0.850 | 15s
  ep  30 | train 0.4615 | val 0.5956 | acc 0.750 tpr 0.650 tnr 0.850 | 15s
  ep  35 | train 0.3968 | val 0.5513 | acc 0.775 tpr 0.750 tnr 0.800 | 15s
  ep  40 | train 0.2829 | val 0.7116 | acc 0.675 tpr 0.700 tnr 0.650 | 15s
  ep  45 | train 0.2365 | val 0.7979 | acc 0.675 tpr 0.550 tnr 0.800 | 15s
  ep  50 | train 0.1566 | val 0.7020 | acc 0.775 tpr 0.800 tnr 0.750 | 15s
  early stop 50, best 35
  >>> TEST Acc=60.0% TPR=55.0% TNR=65.0% | params=85,826 train=12.9min inf=12.09ms 0.41GFLOPs best_ep=35


In [5]:
INCLUDE_SEEDS = [1, 7, 123]

def summarize(rs, name, include=INCLUDE_SEEDS):
    rs = [r for r in rs if r['seed'] in include]
    if not rs: print(f"{name}: no runs"); return
    a = [r['acc'] for r in rs]; t = [r['tpr'] for r in rs]; n = [r['tnr'] for r in rs]
    tm = [r['train_time_sec'] for r in rs]; inf = [r['inf_time_ms'] for r in rs]
    sd = (lambda v: np.std(v, ddof=1)*100 if len(v) > 1 else 0.0)
    print(f"{name}: Acc={np.mean(a)*100:.1f}±{sd(a):.1f}% | "
          f"TPR={np.mean(t)*100:.1f}±{sd(t):.1f}% | TNR={np.mean(n)*100:.1f}±{sd(n):.1f}% | "
          f"Params={rs[0]['n_params']:,} | Train={np.mean(tm)/60:.1f}m | "
          f"Inf={np.mean(inf):.2f}ms | seeds={[r['seed'] for r in rs]}")
    print("    per seed: " + ", ".join(
        f"seed {r['seed']} {r['acc']*100:.1f}%" for r in rs))

print(f"=== v7 ROI Transformer encoder — {TAG}, 200-subject cohort ===")
print("    seeds 1/7/123")
for k, n in [('mri','MRI-only  '), ('pet','PET-only  '), ('mm','Multimodal')]:
    summarize(results[k], n)


with open(RESULTS, 'w') as f:
    json.dump({k: [{kk: (float(vv) if isinstance(vv, (float, np.floating)) else vv)
                    for kk, vv in r.items()} for r in v] for k, v in results.items()},
              f, indent=2)
print(f"\nsaved {RESULTS}")

=== v7 ROI Transformer encoder — tio_tf, 200-subject cohort ===
    seeds 1/7/123
MRI-only  : Acc=52.5±4.3% | TPR=45.0±5.0% | TNR=60.0±10.0% | Params=42,914 | Train=5.3m | Inf=5.24ms | seeds=[1, 7, 123]
    per seed: seed 1 55.0%, seed 7 55.0%, seed 123 47.5%
PET-only  : Acc=67.5±6.6% | TPR=60.0±8.7% | TNR=75.0±5.0% | Params=42,914 | Train=4.2m | Inf=4.50ms | seeds=[1, 7, 123]
    per seed: seed 1 72.5%, seed 7 70.0%, seed 123 60.0%
Multimodal: Acc=58.3±5.2% | TPR=56.7±2.9% | TNR=60.0±13.2% | Params=85,826 | Train=10.8m | Inf=8.30ms | seeds=[1, 7, 123]
    per seed: seed 1 62.5%, seed 7 52.5%, seed 123 60.0%

saved D:/mamba_model/v7_roi_tio_tf_results.json


In [11]:
#  GFLOPs for one forward pass, batch size 1 -- Transformer encoder
#  (6 ROIs, 3,072 tokens per modality)

#  Architecture alone determines these figures, so no training, checkpoints
#  or data are needed. 
import copy
from torch.utils.flop_counter import FlopCounterMode


class _EncStub(nn.Module):
    def __init__(self, log, d, ff, n_layers, batch_first):
        super().__init__()
        self.log, self.d, self.ff = log, d, ff
        self.n_layers, self.bf = n_layers, batch_first
    def forward(self, x, *a, **kw):
        L = x.shape[1] if self.bf else x.shape[0]
        for _ in range(self.n_layers):
            self.log.append((L, self.d, self.ff))
        return torch.zeros_like(x)


def count_flops(model, inputs):
    """Returns (conv_linear, attention, n_tokens, n_layers_seen)."""
    m = copy.deepcopy(model).eval().cpu()
    inputs = [t.detach().cpu() for t in inputs]
    log = []

    def swap(parent):
        for name, child in list(parent.named_children()):
            if isinstance(child, nn.TransformerEncoder):
                lay = child.layers[0]
                setattr(parent, name, _EncStub(
                    log, lay.linear1.in_features, lay.linear1.out_features,
                    len(child.layers),
                    getattr(lay.self_attn, "batch_first", False)))
            else:
                swap(child)
    swap(m)

    counter = FlopCounterMode(display=False)
    with torch.no_grad(), counter:
        m(*inputs)

    cl = counter.get_total_flops()
    at = sum(2 * (4 * L * d * d + 2 * L * L * d + 2 * L * d * ff)
             for L, d, ff in log)
    qkav = sum(2 * (2 * L * L * d) for L, d, ff in log)
    tokens = log[0][0] if log else 0
    del m
    return cl, at, qkav, tokens, len(log)


def report(name, model, inputs):
    p = sum(q.numel() for q in model.parameters() if q.requires_grad)
    cl, at, qkav, tok, nl = count_flops(model, inputs)
    print(f"  {name:32s} {p:>8,}p | {tok:,} tokens | {nl} encoder layers")
    print(f"      conv + linear   {cl/1e9:7.4f}")
    print(f"      attention       {at/1e9:7.4f}   (of which QK^T and AV: {qkav/1e9:.4f})")
    print(f"      total           {(cl+at)/1e9:7.4f} GFLOPs\n")


roi = lambda: torch.randn(1, 6, 1, 64, 64, 64)
print("GFLOPs, one forward pass, batch size 1 -- Transformer encoder\n")
report("Transformer (unimodal)",   TransformerModel().cpu(),           [roi()])
report("Transformer (multimodal)", MultimodalTransformerModel().cpu(), [roi(), roi()])

GFLOPs, one forward pass, batch size 1 -- Transformer encoder

  Transformer (unimodal)             42,914p | 3,072 tokens | 2 encoder layers
      conv + linear    0.1007
      attention        2.5669   (of which QK^T and AV: 2.4159)
      total            2.6676 GFLOPs

  Transformer (multimodal)           85,826p | 3,072 tokens | 4 encoder layers
      conv + linear    0.2013
      attention        5.1338   (of which QK^T and AV: 4.8318)
      total            5.3352 GFLOPs

